S-timator : demonstration of ODE models solving (uses the dynamics.py module).
------------------------------------------------------------------------------

In [1]:
from matplotlib import pyplot as plt
import stimator as st
from stimator.examples import models
st.style.use(['st-seaborn-whitegrid'])

This notebook shows how to use 4 of the most common **S-timator** functions:

- `read_model()`, reads a _string_ that conforms to a model description language, returning a `Model` object
- `solve()`, computes a solution of the ODE system associated with a model.
- `scan()`, calls Model.solve() several times, scanning a model parameter in a range of values.
- `plot()`, draws a graph of the results returned from `solve()` or `scan()`.


### Example 1
Glyoxalase system **model**


In [ ]:
mdl = models.glyoxalases.text
print(mdl)
m1 = st.read_model(mdl)

s = m1.solve(tf=4030.0)
s.plot()

print('==== Last time point ====')
print(f'At t = {s.t[-1]}')
for x in s.last:
    print(f'{x:>8s} = {s.last[x]:f}')

plt.show()

### Example 2
Branched pathway

In [ ]:
from numpy import append, linspace
mdl = models.branched.text

print(mdl)

m2 = st.read_model(mdl)

times = append(linspace(0.0, 5.0, 500), linspace(5.0, 10.0, 500))

m2.solve(tf=10.0, times=times).plot()
plt.show()

### Example 3
Calcium spikes: CICR model

In [ ]:
mdl = models.ca.text

print(mdl)

#chaining functions...
st.read_model(mdl).solve(tf=8.0, npoints=2000).plot(legend='out', xlabel='$t$ (min)')
plt.show()

### Example 4
Rossler chaotic system

In [ ]:
mdl = models.rossler.text
print (mdl)
m4 = st.read_model(mdl)

s = m4.solve(tf=100.0, npoints=2000, outputs="x1 x2 x3".split())

def transformation(vars, t):
    if t > 40.0:
        return (vars[0] - 5.0, vars[1], vars[2])
    else:
        return (-5.0, vars[1], vars[2])

s.apply_transf(transformation)

s.plot()
plt.show()

### Example 5
Lorentz chaotic system: sensitivity to initial conditions

In [ ]:
mdl = models.lorentz.text
print (mdl)
m5 = st.read_model(mdl)

x0s = (1.0, 1.01, 1.02)
titles = [f'$x(0)$ = {x0}' for x0 in x0s]

s = m5.scan({'init.x': x0s}, tf=25.0, npoints=20000, titles=titles)

f, ax = plt.subplots(figsize=(12, 8))
s.one_plot(what='x', ax=ax, label_fmt='$title')
f.suptitle(m5.metadata['title'])
plt.show()

### Example 6
CICR model again: parameter scanning

In [ ]:
m = st.read_model("""
title Calcium Spikes
v0         = -> Ca, 1
v1         = -> Ca, k1*B*step(t, 1.0)
k1         = 7.3
B          = 0.4
export     = Ca ->  , 10 ..
leak       = CaComp -> Ca, 1 ..
!! Ca
v2         = Ca -> CaComp, 65 * Ca**2 / (1+Ca**2)
v3         = CaComp -> Ca, 500*CaComp**2/(CaComp**2+4) * Ca**4/(Ca**4 + 0.6561)
init       : (Ca = 0.1, CaComp = 0.63655)""")

bvalues = (0.0, 0.1, 0.2, 0.25, 0.28, 0.29, 0.3, 0.35,
           0.4, 0.45, 0.5, 0.6, 0.75, 0.8, 0.9, 1.0)
titles = [f'$\\beta$ = {b:g}' for b in bvalues]

s = m.scan({'B': bvalues}, tf=8.0, npoints=1000, titles=titles)
suptitlegend="Dynamics of cytosolic $Ca^{2+}$ as a function of stimulus"

f, axs = st.plots.prepare_grid(s, figsize=(16, 16))

s.plot(ylim=(0, 1.5), axs=axs, legend=False, xlabel='$t$ (min)')
f.suptitle(suptitlegend)
plt.show()

In [ ]:
sols = st.Solutions([s[i] for i in range(0, len(s), 3)])
sols.one_plot(ylim=(0,1.5),
              legend='out',
              xlabel='$t$ (min)',
              label_fmt='$title',
              palette='tab20')
plt.show()

### Example 7

#### A stairway example

In [ ]:
print('Stairway forcing function example')
mtext = """
title a simple 2 enzyme system
v1 : A -> B, rate = Vin*A/(Km + A), V = 0.1, Km = 1
v2 : B -> C, rate = V*B/(Km + B), V = 10, Km = 20
v3 : C ->, rate = kout * C, kout = 1
A = 1

init : B = 0, C = 0

-> Vin = stairway(t, [50, 100, 150, 200, 250], [1, 2, 3, 4, 5])
!! Vin B C
"""
print(mtext)

mstair = st.read_model(mtext)

solstairs = mstair.solve(tf=300, title='stairway')

# f, ax = plt.subplots()

solstairs.plot(legend='out')
plt.show()
